# 01d — Rewarded Ad Engagement

Part 1(d) of the assignment.

This notebook analyses:
1. Share of ad-watching player-days that hit the daily ad cap
2. Total daily rewarded ad views
3. Suspicious ad-view tracking patterns

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.db import get_connection

conn = get_connection()

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Database connected successfully.")

Database connected successfully.


In [2]:
import re

sql_path = "../sql/1d_ad_engagement.sql"

with open(sql_path, "r", encoding="utf-8") as f:
    sql_text = f.read()

# Remove SQL comment lines
sql_clean = re.sub(
    r'^\s*--.*$',
    '',
    sql_text,
    flags=re.MULTILINE
)

# Split separate SQL statements
blocks_01d = [
    query.strip()
    for query in sql_clean.split(";")
    if query.strip()
]

print("SQL queries found:", len(blocks_01d))

assert len(blocks_01d) == 2, \
    f"Expected 2 SQL queries, found {len(blocks_01d)}"

SQL queries found: 2


In [3]:
print(" Cap-hit rate by cap value ")

df_01d_cap = pd.read_sql_query(blocks_01d[0], conn)

display(df_01d_cap)

 Cap-hit rate by cap value 


,ad_daily_cap,ad_watching_player_days,days_hit_cap,pct_hit_cap
0,5,348293,3709,0.0106
1,8,118521,18,0.0002


In [4]:
required_cols = [
    "ad_daily_cap",
    "ad_watching_player_days",
    "days_hit_cap",
    "pct_hit_cap"
]

for col in required_cols:
    assert col in df_01d_cap.columns, \
        f"Missing expected column: {col}"

assert df_01d_cap["pct_hit_cap"].between(0, 1).all(), \
    "ERROR: pct_hit_cap contains values outside [0, 1]"

assert (
    df_01d_cap["days_hit_cap"]
    <= df_01d_cap["ad_watching_player_days"]
).all(), "ERROR: days_hit_cap exceeds player-days"

print("Cap-hit sanity checks passed.")

Cap-hit sanity checks passed.


In [5]:
print(" Daily ad views ")

df_01d_daily = pd.read_sql_query(blocks_01d[1], conn)

print("Shape:", df_01d_daily.shape)

display(df_01d_daily.head(10))

 Daily ad views 
Shape: (120, 3)


,date,total_ad_views,distinct_viewers
0,2026-01-01,536,448
1,2026-01-02,1648,1209
2,2026-01-03,1834,1352
3,2026-01-04,1811,1348
4,2026-01-05,1870,1398
5,2026-01-06,1935,1451
6,2026-01-07,2032,1493
7,2026-01-08,2098,1506
8,2026-01-09,2178,1628
9,2026-01-10,2118,1560


In [6]:
assert len(df_01d_daily) == 120, \
    f"Expected 120 calendar days, found {len(df_01d_daily)}"

assert df_01d_daily["date"].nunique() == 120, \
    "Duplicate or missing dates found"

assert (df_01d_daily["total_ad_views"] >= 0).all()

assert (df_01d_daily["distinct_viewers"] >= 0).all()

print("Daily ad-view checks passed.")

Daily ad-view checks passed.


In [7]:
zero_days = df_01d_daily[
    df_01d_daily["total_ad_views"] == 0
]

print("Days with zero recorded ad views:")
display(zero_days)

Days with zero recorded ad views:


,date,total_ad_views,distinct_viewers
64,2026-03-06,0,0


In [8]:
cap_output = "../outputs/tables/01d_cap_hit_rate.csv"
daily_output = "../outputs/tables/01d_daily_ad_views.csv"

df_01d_cap.to_csv(
    cap_output,
    index=False
)

df_01d_daily.to_csv(
    daily_output,
    index=False
)

print("Saved:", cap_output)
print("Saved:", daily_output)

Saved: ../outputs/tables/01d_cap_hit_rate.csv
Saved: ../outputs/tables/01d_daily_ad_views.csv


In [9]:
saved_cap = pd.read_csv(
    "../outputs/tables/01d_cap_hit_rate.csv"
)

saved_daily = pd.read_csv(
    "../outputs/tables/01d_daily_ad_views.csv"
)

assert len(saved_cap) == len(df_01d_cap)
assert len(saved_daily) == len(df_01d_daily)

assert saved_cap.columns.tolist() == df_01d_cap.columns.tolist()
assert saved_daily.columns.tolist() == df_01d_daily.columns.tolist()

print("Both 01d output files verified successfully.")

Both 01d output files verified successfully.


In [10]:
conn.close()

print("Database connection closed.")

Database connection closed.


## Summary

- For player-days with rewarded-ad activity, 1.06% of cap-5 player-days and 0.02% of cap-8 player-days reached the recorded daily cap. Cap-hitting is therefore uncommon in the observed data, so the results do not provide evidence that the current caps are broadly constraining rewarded-ad engagement.

- The ad-view data contains a systematic daily gap from approximately 08:00–13:59. Other event tables contain activity during these hours, making this pattern more consistent with an ad-specific telemetry or availability issue than normal player behaviour.

- 2026-03-06 contains zero recorded ad views while other game telemetry remains active. The surrounding days are also unusually low, suggesting an ad-specific tracking/availability disruption around this date.

- These gaps were not interpolated or treated as genuine declines in player engagement. They remain visible in the outputs and should be investigated with the ad/analytics pipeline owners.

- `reward_amount` is treated only as an in-game reward and is not interpreted as ad revenue because actual rewarded-ad revenue is not available in the dataset.